In [5]:
import os
import json
import pickle
import base64
from pathlib import Path
from typing import List
from dotenv import load_dotenv

# Unstructured for document parsing
from unstructured.partition.pptx import partition_pptx
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import Element

# LangChain components
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Load environment variables
load_dotenv()

False

In [37]:
!uv remove "unstructured[pptx]"

Resolved 170 packages in 243ms
error: failed to remove file `D:\Projects_Main\ChunkSmith\.venv\Lib\site-packages\debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_cython.cp313-win_amd64.pyd`: Access is denied. (os error 5)


In [24]:
import os
from pathlib import Path
from typing import List

from pptx import Presentation
from unstructured.partition.pptx import partition_pptx

def extract_images_from_pptx(pptx_path: str, output_dir: str):
    # ✅ Ensure output directory exists (REAL FIX)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    print("📁 Image output directory:", output_dir)
    print("📁 Exists:", os.path.exists(output_dir))

    prs = Presentation(pptx_path)
    image_count = 0

    for slide_idx, slide in enumerate(prs.slides):
        for shape_idx, shape in enumerate(slide.shapes):
            # MSO_SHAPE_TYPE.PICTURE == 13
            if shape.shape_type == 13:
                image = shape.image
                ext = image.ext  # png / jpeg

                image_name = f"slide_{slide_idx+1}_img_{shape_idx+1}.{ext}"
                image_path = os.path.join(output_dir, image_name)

                with open(image_path, "wb") as f:
                    f.write(image.blob)

                print(f"✅ Saved image: {image_path}")
                print("   Exists:", os.path.exists(image_path))
                print("   Size:", os.path.getsize(image_path))

                image_count += 1

    print(f"🖼️  Manually extracted {image_count} images from PPTX")


In [25]:
import os
from pathlib import Path
from typing import List
from unstructured.partition.pptx import partition_pptx

def partition_document_launcher(
    file_path: str,
    max_characters: int,
    new_after_n_chars: int,
    combine_text_under_n_chars: int,
    extract_images: bool = False,
    extract_tables: bool = False,
    languages: List[str] = ["eng"]
):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    if max_characters >= new_after_n_chars:
        raise ValueError("max_characters must be less than new_after_n_chars")

    # ✅ USE A REAL, SIMPLE PATH (NO GUESSING)
    image_output_dir = r"D:\Projects_Main\ChunkSmith\extracted_images"

    if extract_images:
        Path(image_output_dir).mkdir(parents=True, exist_ok=True)

    print(f"📄 Partitioning PPTX: {file_path}")
    print(f"⚙️  Images={extract_images}, Tables={extract_tables}")
    print(f"📁 Image dir: {image_output_dir}")

    # ---------- TEXT / TABLE EXTRACTION ----------
    elements = partition_pptx(
        filename=file_path,
        strategy="hi_res",
        hi_res_model_name="yolox",
        chunking_strategy="by_title",
        include_orig_elements=True,
        languages=languages,
        infer_table_structure=extract_tables,
        max_characters=max_characters,
        new_after_n_chars=new_after_n_chars,
        combine_text_under_n_chars=combine_text_under_n_chars,
    )

    print(f"✅ Extracted {len(elements)} text/table elements")

    # ---------- IMAGE EXTRACTION ----------
    if extract_images:
        extract_images_from_pptx(
            pptx_path=file_path,
            output_dir=image_output_dir
        )

    # ---------- ELEMENT BREAKDOWN ----------
    element_types = {}
    for elem in elements:
        name = type(elem).__name__
        element_types[name] = element_types.get(name, 0) + 1

    print(f"📋 Element breakdown: {element_types}")

    return elements


In [26]:
checkpoint1 = partition_document_launcher (file_path =r"D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx",
                                          max_characters=3000,
                                          new_after_n_chars=3800,
                                          combine_text_under_n_chars=200,
                                          extract_images=True,
                                          extract_tables=True,
                                          languages=['eng'],            
                                          )

📄 Partitioning PPTX: D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx
⚙️  Images=True, Tables=True
📁 Image dir: D:\Projects_Main\ChunkSmith\extracted_images
✅ Extracted 23 text/table elements
📁 Image output directory: D:\Projects_Main\ChunkSmith\extracted_images
📁 Exists: True
✅ Saved image: D:\Projects_Main\ChunkSmith\extracted_images\slide_6_img_8.png
   Exists: True
   Size: 22468
✅ Saved image: D:\Projects_Main\ChunkSmith\extracted_images\slide_7_img_10.png
   Exists: True
   Size: 34307
🖼️  Manually extracted 2 images from PPTX
📋 Element breakdown: {'CompositeElement': 23}


In [4]:
checkpoint1[0].metadata.orig_elements

In [6]:
def extract_images_from_pptx(pptx_path: str, output_dir: str) -> List[str]:
    prs = Presentation(pptx_path)
    image_paths = []

    for slide_idx, slide in enumerate(prs.slides):
        for shape_idx, shape in enumerate(slide.shapes):
            if shape.shape_type == 13:  # MSO_SHAPE_TYPE.PICTURE
                image = shape.image
                ext = image.ext
                blob = image.blob

                image_name = f"slide_{slide_idx+1}_img_{shape_idx+1}.{ext}"
                image_path = os.path.join(output_dir, image_name)

                with open(image_path, "wb") as f:
                    f.write(blob)

                image_paths.append(image_path)

    print(f"🖼️  Manually extracted {len(image_paths)} images from PPTX")
    return image_paths


In [8]:
from typing import List, Dict, Any

def images_to_base64_components(image_paths: List[str]) -> List[Dict[str, Any]]:
    image_components = []

    for path in image_paths:
        with open(path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")

        ext = Path(path).suffix.replace(".", "")
        image_components.append({
            "type": "image",
            "filename": Path(path).name,
            "mime_type": f"image/{ext}",
            "base64": encoded
        })

    print(f"🧩 Created {len(image_components)} Base64 image components")
    return image_components


In [23]:
result = partition_document_launcher(
    file_path=r"D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx",
    max_characters=1500,
    new_after_n_chars=1800,
    combine_text_under_n_chars=500,
    extract_images=True,
    extract_tables=True
)

#print("Text elements:", len(result["text_elements"]))
# print("Image elements:", len(result["image_elements"]))


📄 Partitioning PPTX: D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx
⚙️  Images=True, Tables=True
✅ Extracted 13 text/table elements
Print{'D:\\MultiModulRag\\Backend\\SmartPipelinedef\\Images\\slide_6_img_8.png'}
Print{'D:\\MultiModulRag\\Backend\\SmartPipelinedef\\Images\\slide_7_img_10.png'}
🖼️  Manually extracted 2 images from PPTX
📋 Element breakdown: {'CompositeElement': 13}


In [28]:
!uv add comtypes

Resolved 202 packages in 2.85s
Prepared 1 package in 1.08s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 290ms
 + comtypes==1.4.14


In [29]:
import os
import comtypes.client

def pptx_to_pdf(pptx_path: str, pdf_path: str):
    powerpoint = comtypes.client.CreateObject("Powerpoint.Application")
    powerpoint.Visible = 1

    presentation = powerpoint.Presentations.Open(pptx_path)
    presentation.SaveAs(pdf_path, 32)  # 32 = PDF
    presentation.Close()

    powerpoint.Quit()

    print(f"✅ Converted PPTX to PDF: {pdf_path}")


In [30]:
from pathlib import Path
from typing import List
from unstructured.partition.pdf import partition_pdf


d:\Projects_Main\ChunkSmith\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
def process_pdf_with_unstructured(
    pdf_path: str,
    max_characters: int,
    new_after_n_chars: int,
    combine_text_under_n_chars: int,
    extract_images: bool = True,
    extract_tables: bool = True,
    languages: List[str] = ["eng"]
):
    image_output_dir = r"D:\Projects_Main\ChunkSmith\pdf_images"
    Path(image_output_dir).mkdir(parents=True, exist_ok=True)

    print(f"📄 Processing PDF: {pdf_path}")

    elements = partition_pdf(
        filename=pdf_path,
        strategy="hi_res",
        hi_res_model_name="yolox",
        languages=languages,

        extract_images_in_pdf=extract_images,
        extract_image_block_to_payload=extract_images,
        extract_image_block_output_dir=image_output_dir,
        extract_image_block_types=["Image"],

        infer_table_structure=extract_tables,

        chunking_strategy="by_title",
        max_characters=max_characters,
        new_after_n_chars=new_after_n_chars,
        combine_text_under_n_chars=combine_text_under_n_chars,
    )

    print(f"✅ Extracted {len(elements)} elements from PDF")

    # Element breakdown
    breakdown = {}
    for e in elements:
        name = type(e).__name__
        breakdown[name] = breakdown.get(name, 0) + 1

    print(f"📋 Element breakdown: {breakdown}")
    print(f"🖼️ Images saved in: {image_output_dir}")

    return elements


In [32]:
pptx_path = r"D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pptx.pptx"
pdf_path  = r"D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pdf"

# 1️⃣ Convert PPTX → PDF
pptx_to_pdf(pptx_path, pdf_path)

# 2️⃣ Process PDF with Unstructured
elements = process_pdf_with_unstructured(
    pdf_path=pdf_path,
    max_characters=3000,
    new_after_n_chars=3800,
    combine_text_under_n_chars=200,
    extract_images=True,
    extract_tables=True
)


✅ Converted PPTX to PDF: D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pdf
📄 Processing PDF: D:\Projects_Main\ChunkSmith\docs\pdf\Philips-Innovation-That-Matters.pdf


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 16 elements from PDF
📋 Element breakdown: {'CompositeElement': 16}
🖼️ Images saved in: D:\Projects_Main\ChunkSmith\pdf_images


In [36]:
elements[0].metadata.orig_elements[0].to_dict()

{'type': 'Image',
 'element_id': 'cf89a0d9-352b-4826-97c1-1ff437fb1992',
 'text': '',
 'metadata': {'coordinates': {'points': ((np.float64(2500.0),
     np.float64(-0.00033908333326356416)),
    (np.float64(2500.0), np.float64(2249.9996609166665)),
    (np.float64(4000.0), np.float64(2249.9996609166665)),
    (np.float64(4000.0), np.float64(-0.00033908333326356416))),
   'system': 'PixelSpace',
   'layout_width': 4000,
   'layout_height': 2250},
  'last_modified': '2025-12-30T09:53:33',
  'filetype': 'PPM',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg'}}